# AGRIVISION - Plant Detector Training

Trains a custom single-class ("plant") object detector to replace the generic
COCO-SSD model used in the `cv-prototype` demo. Uses a large plant-related subset of
**Open Images V7** (tens of thousands of boxes across Plant / Houseplant / Flower / Tree /
Flowerpot / Fruit / Vegetable categories, merged into one `plant` class) and fine-tunes
**YOLOv8n**, then exports the result to **TensorFlow Lite** (via ONNX + onnx2tf) so it
drops straight into the browser demo's `tfjs-tflite` runtime.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a **GPU** (T4 is fine).

Run all cells top to bottom. At the end you'll download a zip - unzip it into
`agrirover/cv-prototype/model/` on your machine.


In [ ]:
!pip install -q fiftyone ultralytics

## 1. Download a large plant-detection dataset (Open Images V7)

Pulls bounding-box annotations for several plant-related categories. Increase
`MAX_SAMPLES_PER_CLASS` for a larger dataset (more samples = better accuracy, longer download/training).

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

PLANT_CLASSES = [
    "Plant", "Houseplant", "Flower", "Tree", "Flowerpot",
    "Fruit", "Vegetable", "Palm tree",
]
MAX_SAMPLES_PER_CLASS = 1500  # ~10k+ images total across classes; raise for more data

train_ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=PLANT_CLASSES,
    max_samples=MAX_SAMPLES_PER_CLASS,
    dataset_name="agrivision-plants-train",
    shuffle=True,
)

val_ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=PLANT_CLASSES,
    max_samples=max(200, MAX_SAMPLES_PER_CLASS // 5),
    dataset_name="agrivision-plants-val",
    shuffle=True,
)

print(train_ds, val_ds)

## 2. Merge all plant-related labels into a single `plant` class and export as YOLO format

The demo just needs to know "is there a plant here", so we collapse every
plant-related Open Images label onto one class rather than training a multi-class model.

In [ ]:
import fiftyone.utils.yolo as fouy

def remap_to_single_class(dataset):
    for sample in dataset.iter_samples(progress=True, autosave=True):
        dets = sample.ground_truth
        if dets is None:
            continue
        for det in dets.detections:
            det.label = "plant"

remap_to_single_class(train_ds)
remap_to_single_class(val_ds)

EXPORT_DIR = "/content/plant_yolo"

train_ds.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="train",
    classes=["plant"],
)
val_ds.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="val",
    classes=["plant"],
)

print("Exported to", EXPORT_DIR)

## 3. Train YOLOv8n on the plant dataset

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{EXPORT_DIR}/dataset.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project="agrivision",
    name="plant_detector",
)

In [ ]:
metrics = model.val()
print(metrics.box.map50, metrics.box.map)

## 4. Export to TensorFlow Lite for the browser demo


In [ ]:
!pip install -q onnx onnx2tf onnx_graphsurgeon sng4onnx onnxsim onnxruntime

best_weights = "agrivision/plant_detector/weights/best.pt"
export_model = YOLO(best_weights)

# Export to ONNX first, then onnx2tf converts it to TFLite below. NOTE: verified
# by parsing the shipped plant_detector.tflite directly (via the `tflite` flatbuffer
# schema) that onnx2tf's default output for this model keeps the ONNX (NCHW) input
# layout -> [1, 3, imgsz, imgsz], NOT channels-last (NHWC). script.js's
# CustomModel.detect() feeds NCHW to match. If you change onnx2tf's flags/version
# and accuracy craters after a re-export, re-check the input layout before assuming
# it's still NCHW.
onnx_path = export_model.export(format="onnx", imgsz=640, opset=12)


In [ ]:
import json, shutil, glob, subprocess

TFLITE_OUT_DIR = "plant_detector_tflite"

subprocess.run([
    "onnx2tf", "-i", str(onnx_path), "-o", TFLITE_OUT_DIR,
    "-osd",  # output only the plain float32 saved_model/tflite, skip quantized variants
], check=True)

tflite_path = glob.glob(f"{TFLITE_OUT_DIR}/*_float32.tflite")[0]
shutil.copy(tflite_path, f"{TFLITE_OUT_DIR}/plant_detector.tflite")

# Write a small metadata.json the browser app reads directly (avoids parsing
# the yaml ultralytics also emits) so script.js knows the class list and input size.
with open(f"{TFLITE_OUT_DIR}/metadata.json", "w") as f:
    json.dump({"names": ["plant"], "imgsz": 640}, f)

shutil.make_archive("plant_detector_tflite", "zip", TFLITE_OUT_DIR, base_dir=".")
print("Ready: plant_detector_tflite.zip (contains plant_detector.tflite + metadata.json)")


In [ ]:
from google.colab import files
files.download("plant_detector_tflite.zip")


## 5. Install into the demo

Unzip `plant_detector_tflite.zip` and copy `plant_detector.tflite` and `metadata.json`
directly into `agrirover/cv-prototype/model/` on your machine, then reload the AGRIVISION
page. `script.js` checks for `model/plant_detector.tflite` on startup and uses it
automatically, falling back to COCO-SSD if it isn't found.
